# Chains in Langchain: 
hains are fundamental, modular sequences of components (like LLMs, prompts, tools, parsers) linked together to automate multi-step AI workflows, passing output from one step as input to the next, much like a "train of thought" to build complex applications from simple tasks, with common types including LLMChain (basic prompt + model) and SequentialChain (linking multiple sub-chains). 

In [17]:
# model calling through Huggingfacehiub
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
model =ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="openai/gpt-oss-20b",
        task="text-generation",
        huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY"))
    )
model1 =ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id='google/gemma-2-2b-it',
    task='text-generation',
    huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY")
))

# Squenntial chains: 
chain in sequence A--B--C

In [5]:
#Simple Chain 
prompt1 = PromptTemplate(
    template=""" Describe about the following topic in detail:\n
    {topic}
    """, 
    input_variables=["topic"]
)
parser =StrOutputParser()

chain =prompt1 | model | parser
response =chain.invoke({"topic": "Artificial Intelligence"})
print(response)

## Artificial Intelligence (AI) – An In‑Depth Overview

Artificial Intelligence is a broad, interdisciplinary field that seeks to build machines or software capable of performing tasks that, today, require human intelligence. These tasks range from simple pattern recognition to complex decision‑making, natural language understanding, and even creative expression.

Below is a structured exploration of AI, touching on its history, core concepts, technical approaches, real‑world applications, ethical challenges, and future directions.

---

### 1. Foundations

| Concept | What It Is | Key Take‑aways |
|---------|------------|----------------|
| **Intelligence** | Ability to learn, adapt, reason, and solve problems | In AI, “intelligence” is usually measured by performance on benchmark tasks |
| **Artificial** | Constructed by humans (software/hardware) | Distinguishes AI from natural cognition |
| **Goal** | Build systems that can act intelligently in dynamic environments | Requires repre

In [6]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


In [8]:
# sequential Chain

prompt2 = PromptTemplate(
    template=""" Please generate question and answers from given text:\n
    {text}
    """,
    input_variables=["text"]
)
chain2 =prompt1 | model | parser | prompt2 | model | parser

chain2_response =chain2.invoke({"topic": "Machine Learning"})
print(chain2_response)


**Questions & Answers – Machine Learning (ML)**  

1. **What is Machine Learning?**  
   *Answer:*  
   Machine Learning is a subset of artificial intelligence that builds algorithms capable of learning patterns from data rather than being explicitly programmed for every task. It observes data, learns a mapping from inputs to outputs by optimizing a loss function, and then predicts or acts on new, unseen data.

2. **Which three core steps characterize an ML system?**  
   *Answer:*  
   1. **Observe** – Collect inputs (and outputs for supervised tasks).  
   2. **Learn** – Optimize a loss function to create a mapping or internal representation.  
   3. **Predict/Act** – Apply the learned mapping to new data.

3. **Why does ML “turn data into knowledge and knowledge into action”?**  
   *Answer:*  
   Because the learning phase extracts patterns (knowledge) from raw data, and the prediction phase uses that knowledge to make decisions or predictions on new data, effectively turning insig

# Parallel Chain :

```
A--------------------B
   parallel chain        >>merge the chain 
C--------------------D
```

In [15]:
from langchain_openai import OpenAI
load_dotenv()
model2 =OpenAI()

In [16]:
response2 =model2.invoke("Explain about Natural Language Processing in detail.")
print(response2)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# we will create a chain that will create and summery of topic and create job oporutnies and bussiness oportunity for these topics
# we will creater pydantics parser for the final output
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional
class Report(BaseModel):
    report: str = Field(..., description="Detailed report based on the topic summary and business opportunities.")
    best_5_business_opportunities: Optional[List[str]] = Field(..., description="List of the best 5 business opportunities.")

pydanticsparser = PydanticOutputParser(pydantic_object=Report)

prompt3 = PromptTemplate(
    template=""" Generate a detailed summary for the following topic and cgenerate top 5 bussiness opotunity :\n
    {topic}
    """,
    input_variables=["topic"]
)   

prompt4 = PromptTemplate(
    template="""Generate potential job opportunities and business ideas on this topic:\n
    {topic}
    """,
    input_variables=["topic"]
)

prompt5 = PromptTemplate(
    template=""" use topic summary: {summary} and bussiness ooportunities: {bussiness_opportunities} to create a detailed report and top bussiness opportuity.
    \n {format_instructions}
    """,
    input_variables=["summary", "bussiness_opportunities"],
    
    partial_variables={
        "format_instructions": pydanticsparser.get_format_instructions()}
)

parallel_chain = RunnableParallel(
    {'summary': prompt3 | model | parser,
    'bussiness_opportunities': prompt4 | model1 | parser
})
merge_chaion =prompt5 | model | pydanticsparser

final_par_chain =parallel_chain | merge_chaion
final_response =final_par_chain.invoke({"topic": "Artificial Intelligence"})
print(final_response)



report='Artificial Intelligence (AI) has evolved from a theoretical concept in the 1950s to a ubiquitous technology that powers a wide range of modern applications. The field began with foundational ideas such as Alan Turing’s question about machine thought and was formally introduced at the 1956 Dartmouth Conference. Over subsequent decades, AI progressed through rule‑based expert systems, the advent of machine learning and back‑propagation, and landmark achievements like IBM’s Deep Blue and Google’s AlphaGo. The 2010s brought a deep‑learning renaissance with AlexNet, and the 2020s are dominated by large foundation models such as GPT‑4, diffusion models for image generation, and multimodal systems that combine vision, language, and audio.\n\nCore AI paradigms now include symbolic AI, which uses logic and rule‑based reasoning; statistical/connectionist AI, which relies on neural networks; reinforcement learning, which trains agents via reward signals; and hybrid neuro‑symbolic approach

In [20]:
import json 
with open("final_response.json", "w") as f:
    json.dump(final_response.model_dump(), f, indent=4)

In [22]:
print(final_response.report)

Artificial Intelligence (AI) has evolved from a theoretical concept in the 1950s to a ubiquitous technology that powers a wide range of modern applications. The field began with foundational ideas such as Alan Turing’s question about machine thought and was formally introduced at the 1956 Dartmouth Conference. Over subsequent decades, AI progressed through rule‑based expert systems, the advent of machine learning and back‑propagation, and landmark achievements like IBM’s Deep Blue and Google’s AlphaGo. The 2010s brought a deep‑learning renaissance with AlexNet, and the 2020s are dominated by large foundation models such as GPT‑4, diffusion models for image generation, and multimodal systems that combine vision, language, and audio.

Core AI paradigms now include symbolic AI, which uses logic and rule‑based reasoning; statistical/connectionist AI, which relies on neural networks; reinforcement learning, which trains agents via reward signals; and hybrid neuro‑symbolic approaches that me

In [23]:
print(final_response.best_5_business_opportunities)

['AI‑powered Content Creation']


In [24]:
final_par_chain.get_graph().print_ascii()

+------------------------------------------------+   
| Parallel<summary,bussiness_opportunities>Input |   
+------------------------------------------------+   
                 ***            ***                  
               **                  **                
             **                      **              
  +----------------+           +----------------+    
  | PromptTemplate |           | PromptTemplate |    
  +----------------+           +----------------+    
           *                            *            
           *                            *            
           *                            *            
  +-----------------+         +-----------------+    
  | ChatHuggingFace |         | ChatHuggingFace |    
  +-----------------+         +-----------------+    
           *                            *            
           *                            *            
           *                            *            
  +-----------------+       

# Conditional Chains :
